## 授權與來源（License & Attribution）

本課程自編與改編部分：Copyright 2026 leonjyentub；授權：Apache License 2.0（`../LICENSE`）。
上游：Aurélien Géron, `ageron/handson-mlp`（Apache-2.0），commit `47eba45aacc85feae51ba7db68dd1ca66cb25e0a`。
另一來源：`leonjyentub/MachineLearning2025`（課程作者自有程式；其程式另採 Apache-2.0）。
本檔修改聲明：取材與參考 handson-mlp Ch.8；整合 MachineLearning2025 K-means 與半監督示範。 教學版本重新編排與精簡實驗、補充課堂說明及輸出，並非未修改的原始上游檔案。
完整來源對照與第三方授權：`../SOURCES.md`、`../THIRD_PARTY_NOTICES.md`。
上述 Apache-2.0 不延伸至未取得授權的資料、圖片或其他第三方素材；保留所有適用的原始署名。

# 09｜分群與表示應用

對應 `slides/09_分群與表示應用.md`。本檔由 `14_分群與表示應用.ipynb` 改名，程式與已執行輸出保留。

**Colab 示範**：直接上傳此 `.ipynb`，從上到下執行；第一個程式格會安裝缺少的套件，所需資料在使用時自動下載。NumPy 與 scikit-learn 實驗使用 CPU；這些程式沒有可直接切換的 CUDA 後端。輸出存於目前工作目錄。


In [ ]:
# 獨立執行：Colab 上傳此檔後，從這一格依序執行；無需掛載 Git 專案。
import importlib.util, subprocess, sys
_needed = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3", "matplotlib": "matplotlib>=3.7,<4", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2", "cloudpickle": "cloudpickle>=3,<4", "ipywidgets": "ipywidgets>=8,<9"}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib, io, json, os, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# 本機在 programs/ 執行時沿用 data/、outputs/；Colab 則使用自己的工作目錄。
DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
SEED = 42
FAST = os.environ.get("MLCOURSE_PROFILE", "fast") != "full"
_SOURCE_COMMIT = "9e29abbbea6a3ff7250dea64c43c1d6c723095d7"
_SOURCES = {
    "housing.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/housing.tgz", "2364609dc48bec7df3ba9dbb7041478e704ecddcee70ef1827ec3fc49d22c0cc"),
    "lifesat.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/lifesat/lifesat.csv", "e247f7f6c19b0d8c55fb2af293f4fb3b710e31476534ad8e9b7e06ea546a2099"),
}

def setup():
    plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})
    OUT.mkdir(parents=True, exist_ok=True)
    return {"profile": "fast" if FAST else "full", "seed": SEED, "data": str(DATA)}

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

def data_path(name):
    if name not in (*_SOURCES, "mnist_784_v1.npz"):
        raise ValueError(f"未知資料檔：{name}")
    DATA.mkdir(parents=True, exist_ok=True)
    path = DATA / name
    if name in _SOURCES:
        url, expected_sha = _SOURCES[name]
        if not path.exists():
            print(f"下載 {name} …")
            with urllib.request.urlopen(url, timeout=180) as response:
                blob = response.read()
            if name == "housing.csv":
                with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
                    blob = archive.extractfile("housing/housing.csv").read()
            if hashlib.sha256(blob).hexdigest() != expected_sha:
                raise ValueError(f"資料校驗失敗：{name}")
            path.write_bytes(blob)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError(f"本機資料校驗失敗：{name}")
        info = {"url": url, "sha256": expected_sha}
    else:
        if not path.exists():
            print("從 OpenML 下載 MNIST（首次約需數分鐘）…")
            from sklearn.datasets import fetch_openml
            ds = fetch_openml("mnist_784", version=1, as_frame=False,
                              data_home=DATA / "openml_cache", parser="auto")
            X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
            assert X.shape == (70000, 784) and y.shape == (70000,)
            np.savez_compressed(path, X=X, y=y)
        with np.load(path) as archive:
            assert archive["X"].shape == (70000, 784)
            assert archive["y"].shape == (70000,)
        info = {"url": "https://www.openml.org/d/554",
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    manifest_file = DATA / "manifest.json"
    manifest = json.loads(manifest_file.read_text()) if manifest_file.exists() else {}
    manifest[name] = info
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    return path

def housing_split():
    from sklearn.model_selection import train_test_split
    frame = pd.read_csv(data_path("housing.csv"))
    income = pd.cut(frame.median_income, [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
    train, test = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=income)
    assert not set(train.index) & set(test.index)
    return train, test

setup()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. K-means 兩步交替與手算驗算

指派：每點到最近中心；更新：中心移到群平均。對應投影片活動：一維資料 `[0,1,4,5]`、初始中心 `[0,4]`。

In [ ]:
def assign(x, centers):
    return np.argmin(np.abs(np.asarray(x)[:, None] - np.asarray(centers)[None]), axis=1)
def inertia(x, centers, labels):
    return float(np.sum((np.asarray(x) - np.asarray(centers)[labels]) ** 2))

x1 = np.array([0., 1., 4., 5.])
centers0 = np.array([0., 4.])
lab0 = assign(x1, centers0)
inertia_before = inertia(x1, centers0, lab0)
cen1 = np.array([x1[lab0 == k].mean() for k in range(2)])   # 更新步
lab1 = assign(x1, cen1)
inertia_after = inertia(x1, cen1, lab1)
assert lab0.tolist() == [0, 0, 1, 1] and np.isclose(inertia_before, 2.0)
assert np.allclose(cen1, [0.5, 4.5]) and np.isclose(inertia_after, 1.0)
assert lab1.tolist() == lab0.tolist()      # 指派不再改變 -> 收斂
pd.Series({'initial_labels': lab0.tolist(), 'inertia_before': inertia_before,
           'updated_centers': cen1.tolist(), 'inertia_after': inertia_after})

## 2. K-means++、多次初始化與局部解

不同初始化可能收斂到不同局部解（教材圖 8-5）。`init="k-means++"` 讓初始中心分散，`n_init` 取多次最佳（依 inertia）。

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
centers = np.array([[-6, -6], [0, -6], [6, -6], [-3, 4], [4, 4]])
X, y_true = make_blobs(n_samples=800, centers=centers, cluster_std=0.9, random_state=SEED)
runs = []
for seed in range(6):
    km = KMeans(n_clusters=5, init='random', n_init=1, random_state=seed).fit(X)
    runs.append(km.inertia_)
best = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=SEED).fit(X)
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
zz = best.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=.25, cmap='tab10')
plt.scatter(X[:, 0], X[:, 1], c=best.labels_, cmap='tab10', s=10)
plt.scatter(best.cluster_centers_[:, 0], best.cluster_centers_[:, 1], c='k', marker='x', s=120)
plt.title('K-means++ Voronoi regions')
savefig('14_kmeans_voronoi')
print('single random-init inertia range:', round(min(runs), 1), '-', round(max(runs), 1),
      ' | k-means++ x10 inertia:', round(best.inertia_, 1))

## 3. 肘部法與輪廓分數，以及輪廓係數手算

肘部看 inertia 改善開始遞減的位置；輪廓分數把群內距離 $a$ 與最近他群距離 $b$ 一起比較，$s=(b-a)/\max(a,b)$。對應投影片活動的兩點。

In [ ]:
from sklearn.metrics import silhouette_score
def silhouette(a, b):
    return (b - a) / max(a, b)
assert np.isclose(silhouette(2, 5), 0.6) and np.isclose(silhouette(4, 3), -0.25)

ks = range(2, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X).inertia_ for k in ks]
sils = [silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X)) for k in ks]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(ks), inertias, 'o-'); axes[0].set(xlabel='k', ylabel='inertia', title='elbow')
axes[1].plot(list(ks), sils, 'o-'); axes[1].set(xlabel='k', ylabel='mean silhouette', title='silhouette')
savefig('14_kmeans_selection')
best_k = list(ks)[int(np.argmax(sils))]
assert best_k == 5                                   # 資料本來 5 群
print('silhouette picks k =', best_k, ' | hand check: s(a=2,b=5)=0.6, s(a=4,b=3)=-0.25')

## 4. K-means 的限制：橢圓、密度與群大小差異

K-means 偏好相近大小、球狀、密度相近的群（教材圖 8-10）。縮放前要先想清楚「相似」的定義。

In [ ]:
rng = np.random.default_rng(SEED)
blob_a = rng.normal([0, 0], [3.0, 0.4], (400, 2))
blob_b = rng.normal([6, 0], [0.4, 0.4], (60, 2))
Xe = np.vstack([blob_a, blob_b])
km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(Xe)
plt.scatter(Xe[:, 0], Xe[:, 1], c=km.labels_, cmap='coolwarm', s=10)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c='k', marker='x', s=120)
plt.title('elongated + tiny cluster: K-means splits the big blob')
savefig('14_kmeans_limits')
small_recovered = np.bincount(km.labels_[-60:]).max() / 60
print('fraction of the true tiny cluster kept in one K-means label:', round(small_recovered, 2))

## 5. 顏色量化：K-means 當作影像分割

把每個像素當成 RGB 三維點，用 K 個群中心取代所有像素顏色（教材圖 8-11）。使用 scikit-learn 內建 `flower.jpg`。

In [ ]:
from sklearn.datasets import load_sample_image
image = load_sample_image('flower.jpg')
pixels = (image / 255.0).reshape(-1, 3)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image); axes[0].set_title(f'original ({image.shape[0]}x{image.shape[1]})'); axes[0].axis('off')
for ax, n in zip(axes[1:], [8, 4, 2]):
    km = KMeans(n_clusters=n, n_init=3, random_state=SEED).fit(pixels[::5])   # 抽樣加速
    seg = km.cluster_centers_[km.predict(pixels)].reshape(image.shape)
    ax.imshow(seg); ax.set_title(f'{n} colors'); ax.axis('off')
savefig('14_color_quantization')
km8 = KMeans(n_clusters=8, n_init=3, random_state=SEED).fit(pixels[::5])
print('unique colors: original', len(np.unique(pixels, axis=0)), '-> quantized to', km8.n_clusters)

## 6. 少量標註：先標代表樣本，再傳播

digits 只給 50 個標籤預算。比較「隨機 50 個」與「每群挑最接近中心的一張」，再把群中心標籤傳播給整群（教材圖 8-12）。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target, test_size=.3, stratify=digits.target, random_state=SEED)
k = 50
km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xtr)
rep_idx = np.argmin(km.transform(Xtr), axis=0)          # 每群最接近中心的樣本
rep_labels = ytr[rep_idx]

def score(mask_or_idx, y_used):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Xtr[mask_or_idx], y_used)
    return clf.score(Xte, yte)

rng = np.random.default_rng(SEED)
rand_idx = rng.choice(len(Xtr), k, replace=False)
acc_random = score(rand_idx, ytr[rand_idx])
acc_representative = score(rep_idx, rep_labels)
propagated = rep_labels[km.labels_]                     # 把代表標籤傳播給整群
acc_propagated = score(np.arange(len(Xtr)), propagated)
report('clustering', {'kmeans_1d': {'inertia_before': 2.0, 'inertia_after': 1.0},
       'silhouette_best_k': int(best_k), 'label_budget': k,
       'accuracy_random_labels': acc_random, 'accuracy_representative_labels': acc_representative,
       'accuracy_after_propagation': acc_propagated})
assert acc_representative >= acc_random - 0.02
pd.Series({'random_50_labels': acc_random, 'representative_50_labels': acc_representative,
           'propagate_to_full_train': acc_propagated})

## 練習與參考答案

- 群編號 `2` 比 `1` 大嗎？**否，編號沒有大小或固定語義。**
- 平均輪廓分數很高就代表分群成功嗎？**否，要看每群分布與少數群的負值，以及業務意義。**
- 一群只有一個樣本怎麼算輪廓？**沒有同群其他點可平均，通常設 $s=0$。**
- 顏色量化只用 RGB，缺了什麼？**空間位置與語義；相鄰但不同物件的像素可能同群。**